In [2]:
#  external GA workflow for the Binary Adder problem.

from pathlib import Path
import random, csv, json, math, statistics, re, shutil

CONFIG = {
    "seed": 2026,
    "output_root": "binary_adder_ga_external_layer_workflow_corrected",
    "source_candidates": [
        "BinaryAdder_indexed(4).kplt",
        "BinaryAdder_indexed.kplt",
        "../BinaryAdder_indexed(4).kplt",
        "/mnt/data/BinaryAdder_indexed(4).kplt",
    ],
    "population_size": 30,
    "generations": 20,
    "mutation_rate": 0.18,
    "crossover_rate": 0.82,
    "elitism": 3,
    "top_models_to_export": 6,
    "kpworkbench_runs_per_candidate": 10,
    "cases": [
        {"case_id": "case01", "bits_define": 3},
        {"case_id": "case02", "bits_define": 5},
        {"case_id": "case03", "bits_define": 7},
        {"case_id": "case04", "bits_define": 10},
        {"case_id": "case05", "bits_define": 15},
        {"case_id": "case06", "bits_define": 20},
    ],
    "gene_space": {
        "block_size": [1, 2, 3, 4, 5, 8],
        "carry_strategy": ["serial", "block_carry", "lookahead_like", "hybrid"],
        "grouping": [
            "lsb_to_msb",
            "block_grouped",
            "carry_zero_first",
            "carry_one_first",
            "alternating_carry",
            "msb_to_lsb",
        ],
    },
    "fitness_weights": {
        "carry_score": 0.32,
        "block_score": 0.18,
        "grouping_score": 0.15,
        "rule_regular_score": 0.12,
        "scalability_score": 0.15,
        "verification_score": 0.08,
    },
}

random.seed(CONFIG["seed"])

ROOT = Path(CONFIG["output_root"])
CAND_DIR = ROOT / "kplt_candidates"
RES_DIR = ROOT / "ga_results"
LOG_DIR = ROOT / "kpworkbench_logs_to_fill"
for d in [ROOT, CAND_DIR, RES_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def kp_comment(text):
    """Return a kPWorkbench-compatible block comment."""
    return "/* " + str(text).replace("*/", "* /") + " */"

def locate_source():
    for candidate in CONFIG["source_candidates"]:
        p = Path(candidate)
        if p.exists():
            return p
    fallback = ROOT / "BinaryAdder_indexed_fallback.kplt"
    fallback.write_text("""/*
 * BinaryAdder_indexed.kplt
 * Indexed kP-Lingua model for binary addition with carry.
 */
#define bits = 3

type Add {
  max {
    /* carry = 0 */
    x$i$_0, y$i$_0, carry$i$_0 -> sum$i$_0, carry$i+1$_0 . : 0 <= i <= bits
    x$i$_0, y$i$_1, carry$i$_0 -> sum$i$_1, carry$i+1$_0 . : 0 <= i <= bits
    x$i$_1, y$i$_0, carry$i$_0 -> sum$i$_1, carry$i+1$_0 . : 0 <= i <= bits
    x$i$_1, y$i$_1, carry$i$_0 -> sum$i$_0, carry$i+1$_1 . : 0 <= i <= bits

    /* carry = 1 */
    x$i$_0, y$i$_0, carry$i$_1 -> sum$i$_1, carry$i+1$_0 . : 0 <= i <= bits
    x$i$_0, y$i$_1, carry$i$_1 -> sum$i$_0, carry$i+1$_1 . : 0 <= i <= bits
    x$i$_1, y$i$_0, carry$i$_1 -> sum$i$_0, carry$i+1$_1 . : 0 <= i <= bits
    x$i$_1, y$i$_1, carry$i$_1 -> sum$i$_1, carry$i+1$_1 . : 0 <= i <= bits
  }
}

add {
  x0_1, y0_1,
  x1_0, y1_1,
  x2_1, y2_0,
  x3_0, y3_0,
  carry0_0
} (Add) .
""", encoding="utf-8")
    return fallback

SOURCE_PATH = locate_source()
BASE_SOURCE = SOURCE_PATH.read_text(encoding="utf-8")
(ROOT / "binary_adder_ga_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
shutil.copy2(SOURCE_PATH, ROOT / "BinaryAdder_indexed_refined_source.kplt")

def bit_positions(bits_define):
    return bits_define + 1

def bounded_block_sizes(bits_define):
    n = bit_positions(bits_define)
    return [b for b in CONFIG["gene_space"]["block_size"] if b <= n]

def make_gene(bits_define):
    return {
        "block_size": random.choice(bounded_block_sizes(bits_define)),
        "carry_strategy": random.choice(CONFIG["gene_space"]["carry_strategy"]),
        "grouping": random.choice(CONFIG["gene_space"]["grouping"]),
    }

def gene_key(g):
    return (int(g["block_size"]), g["carry_strategy"], g["grouping"])

def estimate_steps(bits_define, g):
    n = bit_positions(bits_define)
    b = max(1, int(g["block_size"]))
    strategy = g["carry_strategy"]
    if strategy == "serial":
        return n + 1
    if strategy == "block_carry":
        return math.ceil(n / b) + b + 1
    if strategy == "lookahead_like":
        return math.ceil(math.log2(max(2, n))) + b + 2
    if strategy == "hybrid":
        return math.ceil(n / b) + math.ceil(math.log2(max(2, b))) + 2
    return n + 2

def proxy_components(bits_define, g):
    n = bit_positions(bits_define)
    b = max(1, int(g["block_size"]))
    strategy = g["carry_strategy"]
    grouping = g["grouping"]

    ideal_block = max(1, round(math.sqrt(n)))
    block_score = max(0.0, 1.0 - abs(b - ideal_block) / max(n, 1))

    strategy_base = {
        "serial": 0.78,
        "block_carry": 0.88,
        "lookahead_like": 0.86,
        "hybrid": 0.93,
    }[strategy]
    if n <= 4 and strategy in {"lookahead_like", "hybrid"}:
        strategy_base -= 0.06
    if n >= 16 and strategy in {"block_carry", "hybrid"}:
        strategy_base += 0.04
    if n >= 16 and strategy == "serial":
        strategy_base -= 0.08
    carry_score = max(0.0, min(1.0, strategy_base))

    grouping_score = {
        "lsb_to_msb": 0.92,
        "block_grouped": 0.95,
        "carry_zero_first": 0.86,
        "carry_one_first": 0.82,
        "alternating_carry": 0.88,
        "msb_to_lsb": 0.70,
    }[grouping]
    if strategy in {"block_carry", "hybrid"} and grouping == "block_grouped":
        grouping_score += 0.03
    if strategy == "serial" and grouping == "lsb_to_msb":
        grouping_score += 0.03
    grouping_score = max(0.0, min(1.0, grouping_score))

    complexity = {
        "serial": 0.05,
        "block_carry": 0.10,
        "lookahead_like": 0.18,
        "hybrid": 0.14,
    }[strategy] + 0.015 * max(0, b - ideal_block)
    rule_regular_score = max(0.0, 1.0 - complexity)

    est_steps = estimate_steps(bits_define, g)
    serial_steps = n + 1
    scalability_score = max(0.0, min(1.0, serial_steps / max(est_steps, 1)))
    if strategy == "serial" and n >= 16:
        scalability_score *= 0.88

    verification_score = 1.0
    if strategy == "lookahead_like":
        verification_score -= 0.10
    if grouping == "msb_to_lsb":
        verification_score -= 0.10
    if b > ideal_block + 3:
        verification_score -= 0.07
    verification_score = max(0.0, min(1.0, verification_score))

    return {
        "carry_score": carry_score,
        "block_score": block_score,
        "grouping_score": grouping_score,
        "rule_regular_score": rule_regular_score,
        "scalability_score": scalability_score,
        "verification_score": verification_score,
        "estimated_steps_proxy": est_steps,
        "ideal_block_size": ideal_block,
    }

def fitness_proxy(bits_define, g):
    c = proxy_components(bits_define, g)
    weights = CONFIG["fitness_weights"]
    score = 100.0 * sum(weights[k] * c[k] for k in weights)
    n = bit_positions(bits_define)
    if n <= 4 and g["carry_strategy"] == "lookahead_like":
        score -= 1.5
    return round(max(0.0, min(100.0, score)), 4), c

def crossover(a, b):
    child = dict(a)
    for key in ["block_size", "carry_strategy", "grouping"]:
        if random.random() < 0.5:
            child[key] = b[key]
    return child

def mutate(g, bits_define):
    g = dict(g)
    if random.random() < CONFIG["mutation_rate"]:
        key = random.choice(["block_size", "carry_strategy", "grouping"])
        if key == "block_size":
            g[key] = random.choice(bounded_block_sizes(bits_define))
        else:
            g[key] = random.choice(CONFIG["gene_space"][key])
    return g

def init_population(bits_define):
    pop = []
    seeds = [
        {"block_size": 1, "carry_strategy": "serial", "grouping": "lsb_to_msb"},
        {"block_size": max(1, round(math.sqrt(bit_positions(bits_define)))), "carry_strategy": "block_carry", "grouping": "block_grouped"},
        {"block_size": max(1, round(math.sqrt(bit_positions(bits_define)))), "carry_strategy": "hybrid", "grouping": "block_grouped"},
        {"block_size": 2, "carry_strategy": "lookahead_like", "grouping": "alternating_carry"},
    ]
    seen = set()
    for s in seeds:
        if s["block_size"] in bounded_block_sizes(bits_define):
            k = gene_key(s)
            if k not in seen:
                pop.append(s); seen.add(k)
    while len(pop) < CONFIG["population_size"]:
        g = make_gene(bits_define)
        if gene_key(g) not in seen:
            pop.append(g); seen.add(gene_key(g))
        elif len(seen) >= len(bounded_block_sizes(bits_define))*len(CONFIG["gene_space"]["carry_strategy"])*len(CONFIG["gene_space"]["grouping"]):
            pop.append(g)
    return pop

def select_tournament(scored):
    group = random.sample(scored, min(3, len(scored)))
    group.sort(key=lambda x: x[0], reverse=True)
    return dict(group[0][1])

def rule_lines_for_constraint(lo, hi):
    suffix = f": {lo} <= i <= {hi}" if lo != hi else f": i = {lo}"
    return {
        "c0_00": f"    x$i$_0, y$i$_0, carry$i$_0 -> sum$i$_0, carry$i+1$_0 . {suffix}",
        "c0_01": f"    x$i$_0, y$i$_1, carry$i$_0 -> sum$i$_1, carry$i+1$_0 . {suffix}",
        "c0_10": f"    x$i$_1, y$i$_0, carry$i$_0 -> sum$i$_1, carry$i+1$_0 . {suffix}",
        "c0_11": f"    x$i$_1, y$i$_1, carry$i$_0 -> sum$i$_0, carry$i+1$_1 . {suffix}",
        "c1_00": f"    x$i$_0, y$i$_0, carry$i$_1 -> sum$i$_1, carry$i+1$_0 . {suffix}",
        "c1_01": f"    x$i$_0, y$i$_1, carry$i$_1 -> sum$i$_0, carry$i+1$_1 . {suffix}",
        "c1_10": f"    x$i$_1, y$i$_0, carry$i$_1 -> sum$i$_0, carry$i+1$_1 . {suffix}",
        "c1_11": f"    x$i$_1, y$i$_1, carry$i$_1 -> sum$i$_1, carry$i+1$_1 . {suffix}",
    }

def blocks(bits_define, block_size, grouping):
    ranges = []
    n = bit_positions(bits_define)
    for lo in range(0, n, block_size):
        hi = min(n - 1, lo + block_size - 1)
        ranges.append((lo, hi))
    if grouping == "msb_to_lsb":
        ranges = list(reversed(ranges))
    return ranges

def ordered_rule_keys(grouping):
    c0 = ["c0_00", "c0_01", "c0_10", "c0_11"]
    c1 = ["c1_00", "c1_01", "c1_10", "c1_11"]
    if grouping == "carry_one_first":
        return c1 + c0
    if grouping == "alternating_carry":
        return [v for pair in zip(c0, c1) for v in pair]
    return c0 + c1

def int_to_bits(value, positions):
    return [(value >> i) & 1 for i in range(positions)]

def deterministic_inputs(bits_define, seed):
    positions = bit_positions(bits_define)
    rng = random.Random(seed)
    max_value = 2 ** positions - 1
    x = rng.randint(0, max_value)
    y = rng.randint(0, max_value)
    return x, y, int_to_bits(x, positions), int_to_bits(y, positions), x + y

def generate_kplt(case_id, candidate_id, bits_define, g, fit, comps, seed):
    positions = bit_positions(bits_define)
    x, y, xb, yb, expected = deterministic_inputs(bits_define, seed)
    input_objs = []
    for i in range(positions):
        input_objs.append(f"x{i}_{xb[i]}")
        input_objs.append(f"y{i}_{yb[i]}")
    input_objs.append("carry0_0")
    input_text = ",\n  ".join(input_objs)

    lines = []
    for lo, hi in blocks(bits_define, int(g["block_size"]), g["grouping"]):
        lines.append("    " + kp_comment(f"GA block {lo}..{hi}; strategy={g['carry_strategy']}; grouping={g['grouping']}"))
        d = rule_lines_for_constraint(lo, hi)
        for key in ordered_rule_keys(g["grouping"]):
            lines.append(d[key])
    rule_text = "\n".join(lines)

    header = kp_comment(
        "GA BINARY ADDER CANDIDATE FOR REAL kPWORKBENCH EVALUATION\n"
        f"candidate_id: {candidate_id}\n"
        f"case_id: {case_id}\n"
        "source_variant: BinaryAdder_indexed_refined\n"
        f"bits_define: {bits_define}\n"
        f"bit_positions: {positions}\n"
        f"chromosome: chi_A=(blockSize={g['block_size']}, carryStrategy={g['carry_strategy']}, grouping={g['grouping']})\n"
        f"fitness_proxy: {fit}\n"
        f"estimated_steps_proxy: {comps['estimated_steps_proxy']}\n"
        f"input_x_decimal: {x}\n"
        f"input_y_decimal: {y}\n"
        f"expected_sum_decimal: {expected}\n"
        "workflow_stage: GA external layer materialized candidate, before real kPWorkbench simulation\n"
        "methodological_note: This file is generated by the external GA layer. It does not contain fabricated real results."
    )

    return f"""{header}

#define bits = {bits_define}

type Add {{
  max {{
{rule_text}
  }}
}}

{kp_comment('Deterministic input, least significant bit first; initial carry is 0')}
add {{
  {input_text}
}} (Add) .
"""

def write_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

history_rows = []
chromosome_rows = []
fitness_rows = []
candidate_rows = []
best_rows = []
plan_rows = []

for case in CONFIG["cases"]:
    case_id = case["case_id"]
    bits_define = int(case["bits_define"])
    pop = init_population(bits_define)
    best_global = None

    for generation in range(CONFIG["generations"]):
        scored = []
        for g in pop:
            fit, comps = fitness_proxy(bits_define, g)
            scored.append((fit, dict(g), comps))
        scored.sort(key=lambda x: x[0], reverse=True)
        best_fit, best_gene, best_comps = scored[0]
        avg_fit = statistics.mean(x[0] for x in scored)
        diversity = len({gene_key(x[1]) for x in scored})
        history_rows.append({
            "case_id": case_id,
            "bits_define": bits_define,
            "bit_positions": bit_positions(bits_define),
            "generation": generation,
            "best_fitness_proxy": best_fit,
            "avg_fitness_proxy": round(avg_fit, 4),
            "population_diversity": diversity,
            "best_block_size": best_gene["block_size"],
            "best_carry_strategy": best_gene["carry_strategy"],
            "best_grouping": best_gene["grouping"],
            "best_estimated_steps_proxy": best_comps["estimated_steps_proxy"],
        })
        if best_global is None or best_fit > best_global[0]:
            best_global = (best_fit, dict(best_gene), dict(best_comps), generation)

        next_pop = [dict(x[1]) for x in scored[:CONFIG["elitism"]]]
        while len(next_pop) < CONFIG["population_size"]:
            p1 = select_tournament(scored)
            p2 = select_tournament(scored)
            child = crossover(p1, p2) if random.random() < CONFIG["crossover_rate"] else dict(p1)
            child = mutate(child, bits_define)
            next_pop.append(child)
        pop = next_pop[:CONFIG["population_size"]]

    final_scored = []
    expanded_pool = [dict(x) for x in pop] + [dict(best_global[1])]
    for block_size in bounded_block_sizes(bits_define):
        for carry_strategy in CONFIG["gene_space"]["carry_strategy"]:
            for grouping in CONFIG["gene_space"]["grouping"]:
                expanded_pool.append({
                    "block_size": block_size,
                    "carry_strategy": carry_strategy,
                    "grouping": grouping,
                })
    seen_pool = set()
    for g in expanded_pool:
        if gene_key(g) in seen_pool:
            continue
        seen_pool.add(gene_key(g))
        fit, comps = fitness_proxy(bits_define, g)
        final_scored.append((fit, dict(g), comps))
    final_scored.sort(key=lambda x: x[0], reverse=True)

    seen = set()
    exported = 0
    for fit, g, comps in final_scored:
        if gene_key(g) in seen:
            continue
        seen.add(gene_key(g))
        exported += 1
        candidate_id = f"{case_id}_cand{exported:03d}"
        kplt_file = CAND_DIR / f"{candidate_id}.kplt"
        seed = CONFIG["seed"] + 1000 * int(case_id.replace("case", "")) + exported
        kplt_file.write_text(generate_kplt(case_id, candidate_id, bits_define, g, fit, comps, seed), encoding="utf-8")

        row = {
            "case_id": case_id,
            "candidate_id": candidate_id,
            "bits_define": bits_define,
            "bit_positions": bit_positions(bits_define),
            "block_size": g["block_size"],
            "carry_strategy": g["carry_strategy"],
            "grouping": g["grouping"],
            "fitness_proxy": fit,
            "carry_score": round(comps["carry_score"], 6),
            "block_score": round(comps["block_score"], 6),
            "grouping_score": round(comps["grouping_score"], 6),
            "rule_regular_score": round(comps["rule_regular_score"], 6),
            "scalability_score": round(comps["scalability_score"], 6),
            "verification_score": round(comps["verification_score"], 6),
            "estimated_steps_proxy": comps["estimated_steps_proxy"],
            "model_file": str(kplt_file),
            "status": "pending_real_kPWorkbench_execution",
        }
        candidate_rows.append(row)
        chromosome_rows.append({
            "case_id": case_id,
            "candidate_id": candidate_id,
            "chromosome": f"chi_A=(blockSize={g['block_size']}, carryStrategy={g['carry_strategy']}, grouping={g['grouping']})",
            **{k: row[k] for k in ["bits_define", "bit_positions", "block_size", "carry_strategy", "grouping", "fitness_proxy", "estimated_steps_proxy", "model_file"]}
        })
        fitness_rows.append({k: row[k] for k in row if k not in {"status"}})

        for run in range(1, CONFIG["kpworkbench_runs_per_candidate"] + 1):
            plan_rows.append({
                "case_id": case_id,
                "candidate_id": candidate_id,
                "run_id": f"run{run:02d}",
                "model_file": str(kplt_file),
                "expected_log_file": f"{candidate_id}_run{run:02d}.out",
                "status": "pending_kPWorkbench_execution",
            })

        if exported >= CONFIG["top_models_to_export"]:
            break

    fit, g, comps, generation = best_global
    best_rows.append({
        "case_id": case_id,
        "bits_define": bits_define,
        "bit_positions": bit_positions(bits_define),
        "best_generation": generation,
        "best_candidate_proxy_after_export": f"{case_id}_cand001",
        "best_fitness_proxy": fit,
        "best_block_size": g["block_size"],
        "best_carry_strategy": g["carry_strategy"],
        "best_grouping": g["grouping"],
        "best_estimated_steps_proxy": comps["estimated_steps_proxy"],
    })

write_csv(RES_DIR / "binary_adder_ga_stage1_generation_history.csv", history_rows)
write_csv(RES_DIR / "binary_adder_ga_stage1_chromosomes.csv", chromosome_rows)
write_csv(RES_DIR / "binary_adder_ga_stage1_fitness_proxy.csv", fitness_rows)
write_csv(RES_DIR / "binary_adder_ga_stage1_candidates.csv", candidate_rows)
write_csv(RES_DIR / "binary_adder_ga_stage1_best_candidates_proxy.csv", best_rows)
write_csv(RES_DIR / "binary_adder_ga_stage2_simulation_plan.csv", plan_rows)

article_section = r"""\subsection{External Genetic Layer for the Refined Binary Adder}
For the refined indexed Binary Adder model, the external genetic layer encodes each candidate as
\[
\chi_A=(blockSize, carryStrategy, grouping).
\]
The proxy fitness is computed before kPWorkbench execution and is therefore not a simulation
result. It estimates the structural quality of the candidate from carry propagation strategy,
block-size adequacy, grouping regularity, rule regularity, scalability, and verification readiness.
The generated candidates are materialized as kP-Lingua files and are subsequently executed in
kPWorkbench to obtain real metrics.
"""
(ROOT / "binary_adder_ga_stage1_article_section.tex").write_text(article_section, encoding="utf-8")

report = [
    "Binary Adder external GA workflow completed.",
    f"Source: {SOURCE_PATH}",
    f"Cases: {len(CONFIG['cases'])}",
    f"Population size: {CONFIG['population_size']}",
    f"Generations: {CONFIG['generations']}",
    f"Candidates exported: {len(candidate_rows)}",
    f"kPWorkbench planned runs: {len(plan_rows)}",
    "Generated .kplt comments use /* ... */ syntax.",
    "No real kPWorkbench results are fabricated here.",
    "",
    "Best proxy candidates:",
]
for row in best_rows:
    report.append(
        f"- {row['case_id']}: {row['best_candidate_proxy_after_export']}, "
        f"bits={row['bits_define']}, blockSize={row['best_block_size']}, "
        f"carryStrategy={row['best_carry_strategy']}, grouping={row['best_grouping']}, "
        f"fitness_proxy={row['best_fitness_proxy']}"
    )
(ROOT / "binary_adder_ga_stage1_report.txt").write_text("\n".join(report), encoding="utf-8")

readme = """# Binary Adder GA external layer workflow

This folder was generated by `ga_binary_adder_external_layer_workflow_corrected.ipynb`.

It contains:
- 36 materialized `.kplt` candidates;
- generation history for the external genetic layer;
- proxy fitness values;
- selected proxy-best candidates;
- a kPWorkbench simulation plan with 10 runs per candidate.

The proxy fitness is a GA-level heuristic computed before real kPWorkbench simulations.
It must be replaced or compared with real fitness after execution logs are collected.
"""
(ROOT / "README.md").write_text(readme, encoding="utf-8")

print("Completed Binary Adder external GA workflow.")
print("Output folder:", ROOT.resolve())
print("Generated .kplt candidates:", len(list(CAND_DIR.glob("*.kplt"))))
print("Planned kPWorkbench runs:", len(plan_rows))
print("CSV results folder:", RES_DIR.resolve())
for r in best_rows:
    print(f"{r['case_id']}: best={r['best_candidate_proxy_after_export']}, "
          f"bits={r['bits_define']}, blockSize={r['best_block_size']}, "
          f"strategy={r['best_carry_strategy']}, grouping={r['best_grouping']}, "
          f"proxy={r['best_fitness_proxy']}")


Completed Binary Adder external GA workflow.
Output folder: C:\Users\student\ICMC2026\binary_adder_ga_external_layer_workflow_corrected
Generated .kplt candidates: 36
Planned kPWorkbench runs: 360
CSV results folder: C:\Users\student\ICMC2026\binary_adder_ga_external_layer_workflow_corrected\ga_results
case01: best=case01_cand001, bits=3, blockSize=2, strategy=block_carry, grouping=block_grouped, proxy=94.66
case02: best=case02_cand001, bits=5, blockSize=2, strategy=hybrid, grouping=block_grouped, proxy=95.78
case03: best=case03_cand001, bits=7, blockSize=3, strategy=hybrid, grouping=block_grouped, proxy=95.78
case04: best=case04_cand001, bits=10, blockSize=3, strategy=hybrid, grouping=block_grouped, proxy=95.78
case05: best=case05_cand001, bits=15, blockSize=4, strategy=hybrid, grouping=block_grouped, proxy=97.06
case06: best=case06_cand001, bits=20, blockSize=5, strategy=hybrid, grouping=block_grouped, proxy=97.06
